# Polymarket News-Shock Pipeline — Colab Runner

Runs **scripts 06–14** to produce `data/analysis/shock_embeddings.parquet`.

**Before opening this notebook:**
1. Zip the project folder: `zip -r thesis-polymarket.zip /path/to/thesis-polymarket/`
2. Upload `thesis-polymarket.zip` to your Google Drive (root or any folder you remember).

**Runtime:** Set to GPU (Runtime → Change runtime type → T4 GPU).

## 1. Mount Drive and extract project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── EDIT THIS if you put the zip in a subfolder ──────────────────────────────
ZIP_PATH = '/content/drive/MyDrive/thesis-polymarket.zip'
# ─────────────────────────────────────────────────────────────────────────────

!unzip -q "{ZIP_PATH}" -d /content/

# Find the extracted root (handles both 'thesis-polymarket/' and flat extraction)
import glob
candidates = glob.glob('/content/thesis-polymarket*')
PROJECT_DIR = [c for c in candidates if os.path.isdir(c)][0]
print('Project dir:', PROJECT_DIR)
os.chdir(PROJECT_DIR)
!ls

## 2. Install dependencies

We install packages manually (no `uv`, no `torch<2.3` Mac constraint) then install the `src/` package with `--no-deps`.

In [ ]:
# Colab already has torch+CUDA — install everything else first
!pip install -q \
    sentence-transformers>=3.0 \
    faiss-cpu \
    httpx>=0.27 \
    pydantic>=2.7 \
    duckdb>=0.10 \
    anthropic>=0.28 \
    google-cloud-bigquery>=3.20 \
    db-dtypes>=1.6.0 \
    feedparser>=6.0 \
    trafilatura>=1.9 \
    newspaper3k>=0.2 \
    pandas>=2.2 \
    pyarrow>=16.0 \
    python-dotenv>=1.0 \
    tenacity>=8.3 \
    "numpy>=1.26,<2" \
    scikit-learn>=1.5 \
    tqdm>=4.66 \
    pandera>=0.19 \
    pyyaml>=6.0

# Install the src/ package without pulling in dependencies again
!pip install -q -e . --no-deps

In [ ]:
# Verify GPU is available
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 3. Environment setup

Only required if you need LLM verification (script 09 with `--llm`) or BigQuery access.
Rule-based verification (default) needs **no API keys**.

In [ ]:
# Optional — only needed if you run script 09 with --llm flag
# import os
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/content/drive/MyDrive/gcp-key.json'

print('Working directory:', os.getcwd())
!ls data/

## 4. Script 06 — Fetch article bodies

Network-fetches full article text for matched candidates. Some URLs will fail (404, paywalls) — that is fine; `body_text_available=False` is acceptable.

In [ ]:
!python scripts/06_fetch_article_bodies.py 2>&1 | tail -30

## 5. Script 07 — BGE matching embeddings

Downloads `BAAI/bge-large-en-v1.5` (~1.3 GB) on first run. Embeds ~475K articles + market questions.
At batch=512 on a T4 this takes **~15 minutes**. Checkpointing is incremental — safe to interrupt and re-run.

In [ ]:
# Restore prior chunks + partial outputs from Drive if present
import os, shutil
from pathlib import Path

DRIVE_EMBS = '/content/drive/MyDrive/thesis-polymarket-results/matching_embeddings'
LOCAL_EMBS = Path('data/news/matching_embeddings')

if os.path.isdir(DRIVE_EMBS):
    LOCAL_EMBS.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_EMBS, str(LOCAL_EMBS), dirs_exist_ok=True)
    print('Restored matching_embeddings from Drive')
    !ls -lh data/news/matching_embeddings/
else:
    print('No prior matching_embeddings on Drive — starting fresh')


In [ ]:
# Run script 07 in bounded pieces, syncing chunks to Drive after each piece.
# Re-opens after Colab disconnects: just re-run this cell.
import subprocess, shutil, os
from pathlib import Path

DRIVE_EMBS = '/content/drive/MyDrive/thesis-polymarket-results/matching_embeddings'
SUCCESS = Path('data/news/matching_embeddings/_SUCCESS')

while not SUCCESS.exists():
    result = subprocess.run(
        ['python', 'scripts/07_embed_for_matching.py', '--batch-size', '512', '--max-chunks', '10'],
        capture_output=False,
    )
    # Sync current state (chunks + any merged parquet) to Drive
    os.makedirs(DRIVE_EMBS, exist_ok=True)
    shutil.copytree('data/news/matching_embeddings', DRIVE_EMBS, dirs_exist_ok=True)
    print('Synced to Drive.')
    if result.returncode != 0:
        raise RuntimeError(f"script 07 exited with code {result.returncode} — see output above")
    if SUCCESS.exists():
        print('All done — _SUCCESS written.')
        break
    print('Not done yet — continuing...')

## 6. Script 08 — FAISS match candidates

In [ ]:
!python scripts/08_match_candidates.py 2>&1

## 7. Script 09 — Rule-based match verification

Default (no `--llm`) uses the fast rule-based verifier — no API key needed.

In [ ]:
!python scripts/09_llm_verify_matches.py 2>&1

## 8. Script 10 — Deduplicate into news events

In [ ]:
!python scripts/10_dedup_into_events.py 2>&1

## 9. Script 11 — E5 analysis embeddings

Downloads `intfloat/e5-large-v2` (~1.2 GB) on first run. Embeds only the verified articles (much smaller set than 475K). ~5 minutes on T4.

In [ ]:
!python scripts/11_embed_for_analysis.py --batch-size 256 2>&1

## 10. Script 12 — Assemble dataset tuples

In [ ]:
!python scripts/12_assemble_dataset.py 2>&1

## 11. Script 13 — Purge correlates and compute shocks

In [ ]:
!python scripts/13_purge_and_compute_shocks.py 2>&1

## 12. Script 14 — Feasibility gate

Checks density thresholds. Prints a pass/fail report.

In [ ]:
!python scripts/14_feasibility_gate.py 2>&1

## 13. Verify outputs

In [ ]:
import pandas as pd
from pathlib import Path

shock_path = Path('data/analysis/shock_embeddings.parquet')
tuples_path = Path('data/analysis/tuples.parquet')
success = Path('data/analysis/_FOCAL_SUCCESS')

print('_FOCAL_SUCCESS exists:', success.exists())
print()

if shock_path.exists():
    df = pd.read_parquet(shock_path)
    print('shock_embeddings.parquet shape:', df.shape)
    print('columns:', df.columns.tolist())
    print()
    print(df.groupby(['category', 'split']).size().to_string())
else:
    print('shock_embeddings.parquet NOT FOUND — check script 13 output above')

## 14. Save results back to Drive

Copy the `data/analysis/` outputs back to Google Drive so you can download them.

In [ ]:
DRIVE_OUTPUT = '/content/drive/MyDrive/thesis-polymarket-results'
!mkdir -p "{DRIVE_OUTPUT}"
!cp -r data/analysis/ "{DRIVE_OUTPUT}/"
!cp -r data/matches/ "{DRIVE_OUTPUT}/"
!cp -r data/news/analysis_embeddings/ "{DRIVE_OUTPUT}/" 2>/dev/null || true
!cp -r data/news/matching_embeddings/ "{DRIVE_OUTPUT}/" 2>/dev/null || true
print('Saved to', DRIVE_OUTPUT)
!ls "{DRIVE_OUTPUT}"